<a href="https://colab.research.google.com/github/aparimi123/ITCS-3162/blob/main/lab_03_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Exercise: Wrangling the Google Play Store Dataset

**ITCS 3162 — Introduction to Data Mining**

**Name:** _Abhi Parimi_
**Date:** _05/29/2026_

This dataset is *messier* than diamonds — it's scraped from the Google Play Store and has the kinds of problems you'll see in the wild: numbers stored as strings, units mixed with values, duplicates, weird placeholder strings, and missing values. That's the point: you'll do real cleaning.

**Source:** Kaggle's "Google Play Store Apps" dataset (mirrored on GitHub for stable URL access).

When you're done, **Restart & Run All**, download as `.ipynb`, and submit via Canvas.


## Setup

Run this cell. It loads the dataset directly from a GitHub URL.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/sumitgirwal/google-play-store-data-analysis/master/googleplaystore.csv"
df = pd.read_csv(URL)
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Exercise 1 — Initial inspection (10 pts)

In the cell below, print **all three**:
1. The shape of the DataFrame (rows, columns)
2. The output of `df.info()`
3. The output of `df.describe(include="all")`

Then in the markdown cell, answer the questions.


In [ ]:
# TODO: print shape, info(), describe(include='all')
print(df.shape)
print(df.info())
print(df.describe(include='all'))



(10841, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB
None
           App Category       Rating Reviews                Size    Installs  \
count    10841    10841  9367.000000   10841               10841  

**Answer the following:**

1. How many rows and columns does the dataset have? **5rows and 13 columns**

2. Look at the `Reviews`, `Size`, `Installs`, and `Price` columns. What dtype does pandas assign them? Why is that a problem for a numeric column like `Reviews`? **It assigns them object. This is a problem because you can't calculate averages, make numerical plots, and more. **

3. Which columns appear to have missing values? **Rating, Size, Current Ver, Android Ver**

YOUR ANSWERS:



## Exercise 2 — Missing values (10 pts)

In the cell below, compute (a) the count of missing values per column, sorted descending, and (b) the **percentage** of missing values per column rounded to 2 decimal places. Then drop any row that is missing the `Rating` column (since it's the most important missingness to handle) and store the result in `df_r`.


In [ ]:
# TODO: counts of missing values per column (sorted descending)
df.isna().sum().sort_values(ascending=False)


# TODO: percentage missing per column (rounded to 2 decimal places)
(df.isna().mean().round(2) * 100)


# TODO: create df_r by dropping rows where Rating is NaN; print its new shape
df_r = df.dropna(subset=["Rating"])
df_r.shape


(9367, 13)

## Exercise 3 — Duplicates (10 pts)

There are duplicate rows in this dataset (the same app appears multiple times). In the cell below:
1. Print how many fully-duplicate rows exist in `df_r`.
2. How many duplicate values are there in the `App` column specifically (apps appearing under different categories)?
3. Create `df_dedup` by dropping fully-duplicate rows and print its new shape.


In [ ]:
# TODO: fully duplicate rows
df_r.duplicated().sum()


# TODO: duplicate App names (use df_r["App"].duplicated().sum())
df_r["App"].duplicated().sum()


# TODO: df_dedup = df_r.drop_duplicates(); print shape
df_dedup = df_r.drop_duplicates()
df_dedup.shape


(8893, 13)

## Exercise 4 — Cleaning the `Installs` column (15 pts)

The `Installs` column looks like `"1,000,000+"` — a string with a `+` and commas. Convert it to an integer column. Steps:

1. Inspect a few unique values: `df_dedup["Installs"].unique()[:10]`
2. Remove `+` and `,` characters with `.str.replace()`
3. Convert to int with `.astype(int)` (or use `pd.to_numeric` with `errors="coerce"` if any values look weird)
4. Assign the cleaned values back to `df_dedup["Installs"]`
5. Verify with `df_dedup["Installs"].dtype` and `.head()`


In [ ]:
# TODO: inspect unique values
df_dedup["Installs"].unique()[:20]

# TODO: clean Installs to integer
df_dedup["Installs"] = (
    df_dedup["Installs"]
        .str.replace("+", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace("Free", "0")      # fixes the ValueError
        .astype(int)
)

# verify
df_dedup["Installs"].dtype, df_dedup["Installs"].head()


/tmp/ipykernel_421/2050934824.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dedup["Installs"] = (


(dtype('int64'),
 0       10000
 1      500000
 2     5000000
 3    50000000
 4      100000
 Name: Installs, dtype: int64)

## Exercise 5 — Cleaning the `Price` column (15 pts)

`Price` looks like `"0"` for free apps and `"$4.99"` for paid ones. Convert it to a float.

1. Use `.str.replace("$", "", regex=False)` then `.astype(float)`.
2. Watch for any unexpected values (the real dataset has at least one weird row — the column may have a non-numeric placeholder that `pd.to_numeric(..., errors="coerce")` handles gracefully).
3. After cleaning, print how many paid apps there are (where `Price > 0`).


In [ ]:
# TODO: clean Price to float
df_dedup["Price"] = (
    df_dedup["Price"]
        .str.replace("$", "", regex=False)
        .replace("Everyone", "0")
)
df_dedup["Price"] = pd.to_numeric(df_dedup["Price"], errors="coerce")

# TODO: count paid apps
(df_dedup["Price"] > 0).sum()



/tmp/ipykernel_421/358648824.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dedup["Price"] = (
/tmp/ipykernel_421/358648824.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dedup["Price"] = pd.to_numeric(df_dedup["Price"], errors="coerce")


np.int64(613)

## Exercise 6 — Filtering and a derived column (15 pts)

1. Create `top_apps`, containing only apps with `Rating >= 4.5` and `Installs >= 1_000_000`.
2. Print its shape and the top 5 categories of those apps by count.
3. In `df_dedup`, add a new column `Revenue` defined as `Price * Installs` (zero for free apps).
4. Show the top 10 apps by `Revenue`.


In [ ]:
# TODO: top_apps and category counts
top_apps = df_dedup[(df_dedup["Rating"] >= 4.5) & (df_dedup["Installs"] >= 1_000_000)]
top_apps.shape, top_apps["Category"].value_counts().head()

# TODO: Revenue column and top 10
df_dedup["Revenue"] = df_dedup["Price"] * df_dedup["Installs"]
df_dedup.sort_values("Revenue", ascending=False).head(10)


/tmp/ipykernel_421/2812161456.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dedup["Revenue"] = df_dedup["Price"] * df_dedup["Installs"]


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Revenue
4347,Minecraft,FAMILY,4.5,2375336,Varies with device,10000000,Paid,6.99,Everyone 10+,Arcade;Action & Adventure,"July 24, 2018",1.5.2.1,Varies with device,69900000.0
2241,Minecraft,FAMILY,4.5,2376564,Varies with device,10000000,Paid,6.99,Everyone 10+,Arcade;Action & Adventure,"July 24, 2018",1.5.2.1,Varies with device,69900000.0
5351,I am rich,LIFESTYLE,3.8,3547,1.8M,100000,Paid,399.99,Everyone,Lifestyle,"January 12, 2018",2.0,4.0.3 and up,39999000.0
5356,I Am Rich Premium,FINANCE,4.1,1867,4.7M,50000,Paid,399.99,Everyone,Finance,"November 12, 2017",1.6,4.0 and up,19999500.0
4034,Hitman Sniper,GAME,4.6,408292,29M,10000000,Paid,0.99,Mature 17+,Action,"July 12, 2018",1.7.110758,4.1 and up,9900000.0
7417,Grand Theft Auto: San Andreas,GAME,4.4,348962,26M,1000000,Paid,6.99,Mature 17+,Action,"March 21, 2015",1.08,3.0 and up,6990000.0
2883,Facetune - For Free,PHOTOGRAPHY,4.4,49553,48M,1000000,Paid,5.99,Everyone,Photography,"July 25, 2018",1.3.1,4.1 and up,5990000.0
5578,Sleep as Android Unlock,LIFESTYLE,4.5,23966,872k,1000000,Paid,5.99,Everyone,Lifestyle,"June 27, 2018",20180608,4.0 and up,5990000.0
8804,DraStic DS Emulator,GAME,4.6,87766,12M,1000000,Paid,4.99,Everyone,Action,"July 19, 2016",r2.5.0.3a,2.3 and up,4990000.0
4367,I'm Rich - Trump Edition,LIFESTYLE,3.6,275,7.3M,10000,Paid,400.00,Everyone,Lifestyle,"May 3, 2018",1.0.1,4.1 and up,4000000.0


## Exercise 7 — Group-by summary (15 pts)

Using `df_dedup`, produce a single summary table with one row per **Category**, containing:
- `n_apps` — number of apps in that category
- `mean_rating` — average rating (rounded to 2 decimals)
- `mean_installs` — average installs (rounded to 0 decimals)
- `total_revenue` — sum of `Revenue`

Sort by `total_revenue` descending. Show only the top 10 rows.


In [ ]:
# TODO: groupby summary
summary = (
    df_dedup
        .groupby("Category")
        .agg(
            n_apps=("App", "count"),
            mean_rating=("Rating", lambda x: round(x.mean(), 2)),
            mean_installs=("Installs", lambda x: round(x.mean(), 0)),
            total_revenue=("Revenue", "sum")
        )
        .sort_values("total_revenue", ascending=False)
        .head(10)
)

summary



,n_apps,mean_rating,mean_installs,total_revenue
Category,,,,
FAMILY,1718,4.19,5844663.0,1.857743e+08
LIFESTYLE,305,4.10,1753250.0,5.758394e+07
GAME,1074,4.28,29370449.0,4.098684e+07
FINANCE,317,4.13,2430008.0,2.572664e+07
PHOTOGRAPHY,304,4.18,31977773.0,8.941050e+06
MEDICAL,302,4.18,139612.0,8.371355e+06
PERSONALIZATION,310,4.33,6691461.0,7.786310e+06
TOOLS,734,4.05,15600442.0,5.462910e+06
SPORTS,286,4.23,5344516.0,4.706154e+06


## Exercise 8 — Reflection (10 pts)

In 4–6 sentences, answer all of:

1. Which cleaning step would have been impossible to skip if you wanted to compute average install counts? Why?
**A cleaning step that would have been impossible to skip is fixing the installs column because you can't compute an average if the values still contain things such as commas, plus signs, and more**
2. What's one piece of information in the raw data that you *could* extract with more work but didn't (e.g., the `Size` column, the `Last Updated` column)? How would you approach it?
**A piece of information which I didn't fully extract is the Size column. I would approach it by converting the units into a consistent numeric format**
3. Did anything in the data surprise you?
**I was suprised by how many incorrect entries there were. I expected some but there were a substantial amount**




## Submission checklist

- [ ] Name and date filled in
- [ ] All TODO cells completed and run
- [ ] All `YOUR ANSWER` prompts replaced
- [ ] **Restart & Run All** completes without errors
- [ ] Downloaded as `.ipynb` and uploaded to Canvas
